# 1.0 Importando pacotes

In [1]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
import os

# 2.0 Definindo caminhos

In [2]:
# Caminho da pasta de inputs
inputs_path = Path('./inputs').resolve()

# Caminho da pasta de outputs
outputs_path = Path('./outputs').resolve()

# Caminhos dos subsets
co_path = inputs_path / 'buffered_subset_co.parquet'
no2_path = inputs_path / 'buffered_subset_no2.parquet'
o3_path = inputs_path / 'buffered_subset_o3.parquet'
pm_path = inputs_path / 'buffered_subset_pm.parquet'
so2_path = inputs_path / 'buffered_subset_so2.parquet'


# 3.0 Carregando subsets

In [3]:
buffered_subset_co = gpd.read_parquet(co_path)
buffered_subset_no2 = gpd.read_parquet(no2_path)
buffered_subset_o3 = gpd.read_parquet(o3_path)
buffered_subset_pm = gpd.read_parquet(pm_path)
buffered_subset_so2 = gpd.read_parquet(so2_path)

# 4.0 Formatando outputs

In [4]:
# 1) estacoes_completa --------------------------------------------------------------
# Unindo poluentes em um gdf final completo
buffered_stations = pd.concat([buffered_subset_co,
                               buffered_subset_no2,
                               buffered_subset_o3,
                               buffered_subset_pm,
                               buffered_subset_so2])


# 2) rep_espacial -------------------------------------------------------------------
"""
Planilha original de estações de monitoramento + colunas [REP_ESPACIAL_NAME] e [REP_ESPACIAL]
    - REP_ESPACIAL_NAME: nome da classe de representatividade espacial (micro, meso, 
    bairro, urbana)
    - REP_ESPACIAL: tamanho do raio do buffer de representatividade em metros
"""
# Removendo colunas auxiliares
def drop_aux_cols(subset):
    return subset.drop(columns= (list(subset
                                      .filter(like='rep')
                                      .columns) +
                                 list(subset
                                      .filter(like='min')
                                      .columns) +
                                 list(subset
                                      .filter(like='max')
                                      .columns) +
                                 list(subset
                                      .filter(like='k')
                                      .columns) +
                                 list(subset
                                      .filter(like='micro')
                                      .columns) +
                                 list(subset
                                      .filter(like='meso')
                                      .columns) +
                                 list(subset
                                      .filter(like='bairro')
                                      .columns) +
                                 list(subset
                                      .filter(like='urb')
                                      .columns) +
                                 ['EPSG','distance_to_industry',
                                  'industry_geom']
                                 )
                       )
# Aplicando função
filtered_stations = drop_aux_cols(buffered_stations) 


# 5.0 Salvando outputs

In [5]:
# Salvando o GeoDataFrame com a indústria mais próxima de cada estação
buffered_stations.to_parquet(outputs_path / 'estacoes_completa.parquet')

# Salvando o GeoDataFrame de input com as estações, sua classificação de 
# representatividade espacial e o tamanho do buffer
filtered_stations.to_csv(outputs_path / 'rep_espacial.csv')

In [6]:
filtered_stations.columns

Index(['UF', 'CIDADE', 'CD_MUN', 'ID_OEMA', 'ID_MMA', 'ID_MMA_COMPLETO',
       'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE',
       'FUNCIONAMENTO', 'CATEGORIA', 'METODO', 'CALIBRACAO', 'MARCA', 'MODELO',
       'POLUENTE', 'COD_POLUENTE', 'MOBILIDADE', 'FINALIDADE', 'STATUS',
       'INICIO', 'FIM', 'LATITUDE', 'LONGITUDE', 'MONITORAR', 'FONTE',
       'CERTIFICACAO', 'COD_UF_IBGE', 'ANOS_MONITORADOS', 'BASE_DADOS',
       'ELEVACAO', 'REALOCACAO', 'OBS_CALIBRACAO', 'DADOS_MONITORAMENTO',
       'RECONHECIDA', 'OBS_GERAIS', 'REP_ESPACIAL_DECLARADA', 'OPERACAO',
       'geometry', 'REP_ESPACIAL_NAME', 'osm_id_mais_prox_valida',
       'REP_ESPACIAL'],
      dtype='object')

In [7]:
filtered_stations

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,REP_ESPACIAL_DECLARADA,OPERACAO,geometry,REP_ESPACIAL_NAME,osm_id_mais_prox_valida,REP_ESPACIAL
0,MA,São Luís,2111300,Santa Bárbara,MA0001,MA0001RA007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,POINT (-44.21183 -2.5953),Bairro,665480904,4000
1,RJ,Rio de Janeiro,3304557,RJ - Taquara,RJ0021,RJ0021RA007,INEA,Publica,Ambiental RB,Privada,...,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,POINT (-43.37174 -22.93466),Bairro,1265877284,4000
2,MA,São Luís,2111300,Anjo da Guarda,MA0002,MA0002RA007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,POINT (-44.34029 -2.56118),Bairro,46424460,4000
3,MA,São Luís,2111300,UTEInterna,MA1001,MA1001RA007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,Nao reconhecida,Nao declarado,Nao declarado,Nao declarado,POINT (-44.33979 -2.58613),Bairro,778171754,4000
4,RJ,Rio de Janeiro,3304557,RJ - Van (Sumare-SBT),RJ0218,RJ0218RA007,INEA,Publica,Ambiental RB,Privada,...,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,Nao declarado,POINT (-43.22944 -22.94972),Bairro,1027215128,4000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,RS,Canoas,4304606,Canoas VCOMAR,RS0004,RS0004RA003,FEPAM,Publica,FEPAM,Publica,...,Nao declarado,Sim,Nao declarado,Nao declarado,Bairro,Nao declarado,POINT (-51.18179 -29.9303),Bairro,41809547,4000
209,RS,Canoas,4304606,Esteio Vila Ezequiel,RS0006,RS0006RA003,Refap,Publica,Refap,Publica,...,Nao declarado,Sim,Nao declarado,Nao declarado,Urbana,Nao declarado,POINT (-51.18179 -29.9303),Bairro,41809547,4000
210,PR,MARINGÁ,4115200,MRGA,PR0013,PR0013RA003,IAT,Publica,IAT,Publica,...,Nao declarado,Nao declarado,Nao declarado,"Estação com implantação de MP2,5",Bairro,Nao declarado,POINT (-51.93824 -23.41558),Bairro,232493474,4000
211,PR,CURITIBA,4106902,BOQ,PR0004,PR0004RA003,IAT,Publica,IAT,Publica,...,Nao declarado,Nao declarado,Nao declarado,Estação nova em fase de implementação na rede,Bairro,Nao declarado,POINT (-49.2458 -25.49317),Bairro,33068535,4000
